In [1]:
import sys
import csv

from sklearn.svm import SVC
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB 
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

csv.field_size_limit(2**31-1)

131072

In [2]:
def get_data(t='test'):
    text= []
    label= []

    with open(f'../data-vermeer/{t}.csv') as fi:
        next(fi) # skips header row
        reader = csv.reader(fi, delimiter=',')

        for row in reader:
            text.append(row[0])
            label.append(row[1])

    return text, label

In [3]:
X_test, y_test = get_data('test')
X_train, y_train = get_data('train')

In [4]:
print(len(X_test), len(y_test))
print(len(X_train), len(y_train))   

698 698
2788 2788


In [6]:
configurations = [('NB with Count', CountVectorizer(min_df=5, max_df=.5), MultinomialNB()),
                 ('NB with TfIdf', TfidfVectorizer(min_df=5, max_df=.5), MultinomialNB()),
                 ('LogReg with Count', CountVectorizer(min_df=5, max_df=.5), LogisticRegression(solver='liblinear')),
                 ('LogReg with TfIdf', TfidfVectorizer(min_df=5, max_df=.5), LogisticRegression(solver='liblinear')),
                 ('SVM with Count - rbf kernel', CountVectorizer(min_df=5, max_df=.5), SVC(kernel='rbf')),
                 ('SVM with Count - linear kernel', CountVectorizer(min_df=5, max_df=.5), SVC(kernel='linear')),
                 ('SVM with Tfidf - rbf kernel', TfidfVectorizer(min_df=5, max_df=.5), SVC(kernel='rbf')),
                 ('SVM with Tfidf - linear kernel', TfidfVectorizer(min_df=5, max_df=.5), SVC(kernel='linear')),
            # Added Random Forest classifiers:
                 ('Random Forest with Count', CountVectorizer(min_df=5, max_df=.5), RandomForestClassifier(n_estimators=100, random_state=42)),
                 ('Random Forest with TfIdf', TfidfVectorizer(min_df=5, max_df=.5), RandomForestClassifier(n_estimators=100, random_state=42)),
                 ]

for description, vectorizer, classifier in configurations:
    print(description)
    X_tr = vectorizer.fit_transform(X_train)
    X_te = vectorizer.transform(X_test)
    classifier.fit(X_tr, y_train)
    y_pred = classifier.predict(X_te)
    print(metrics.classification_report(y_test, y_pred) )
    print('\n')

NB with Count
               precision    recall  f1-score   support

     business       0.48      0.76      0.59       101
entertainment       0.95      0.73      0.83       400
        other       0.52      0.58      0.55        73
     politics       0.71      0.85      0.78       124

     accuracy                           0.74       698
    macro avg       0.66      0.73      0.68       698
 weighted avg       0.79      0.74      0.75       698



NB with TfIdf
               precision    recall  f1-score   support

     business       0.80      0.16      0.26       101
entertainment       0.67      0.99      0.80       400
        other       1.00      0.11      0.20        73
     politics       0.90      0.53      0.67       124

     accuracy                           0.70       698
    macro avg       0.84      0.45      0.48       698
 weighted avg       0.76      0.70      0.64       698



LogReg with Count


c:\Users\stolw010\AppData\Local\anaconda3\envs\gesis_iml\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


               precision    recall  f1-score   support

     business       0.63      0.56      0.59       101
entertainment       0.83      0.94      0.88       400
        other       0.74      0.51      0.60        73
     politics       0.90      0.77      0.83       124

     accuracy                           0.81       698
    macro avg       0.77      0.69      0.73       698
 weighted avg       0.80      0.81      0.80       698



LogReg with TfIdf


c:\Users\stolw010\AppData\Local\anaconda3\envs\gesis_iml\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


               precision    recall  f1-score   support

     business       0.80      0.47      0.59       101
entertainment       0.75      0.98      0.85       400
        other       0.89      0.34      0.50        73
     politics       0.90      0.65      0.75       124

     accuracy                           0.78       698
    macro avg       0.84      0.61      0.67       698
 weighted avg       0.80      0.78      0.76       698



SVM with Count - rbf kernel
               precision    recall  f1-score   support

     business       0.72      0.29      0.41       101
entertainment       0.68      0.99      0.81       400
        other       1.00      0.12      0.22        73
     politics       0.93      0.52      0.66       124

     accuracy                           0.71       698
    macro avg       0.83      0.48      0.53       698
 weighted avg       0.77      0.71      0.66       698



SVM with Count - linear kernel
               precision    recall  f1-score   supp

#### now add a lemmatizer:
(Note I only keep the best of previous runs to save compute, is that smart?)

In [6]:
# Import spaCy and load the English model
import spacy
nlp = spacy.load('en_core_web_sm')

# Define a spaCy lemmatizer tokenizer
def spacy_lemmatizer(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc]

configurations = [
    ('LogReg with Count', CountVectorizer(min_df=5, max_df=0.5), LogisticRegression(solver='saga')),
    ('SVM with Tfidf - linear kernel', TfidfVectorizer(min_df=5, max_df=0.5), SVC(kernel='linear')),
    # Add spaCy lemmatizer options
    ('LogReg with Count + spaCy lemmatizer', CountVectorizer(min_df=5, max_df=0.5, tokenizer=spacy_lemmatizer), LogisticRegression(solver='saga')),
    ('SVM with Tfidf - linear kernel + spaCy lemmatizer', TfidfVectorizer(min_df=5, max_df=0.5, tokenizer=spacy_lemmatizer), SVC(kernel='linear')),
]

for description, vectorizer, classifier in configurations:
    print(description)
    X_tr = vectorizer.fit_transform(X_train)
    X_te = vectorizer.transform(X_test)
    classifier.fit(X_tr, y_train)
    y_pred = classifier.predict(X_te)
    print(metrics.classification_report(y_test, y_pred))
    print('\n')

LogReg with Count


c:\Users\stolw010\AppData\Local\anaconda3\envs\gesis_iml\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


               precision    recall  f1-score   support

     business       0.61      0.50      0.55       101
entertainment       0.77      0.95      0.85       400
        other       0.79      0.21      0.33        73
     politics       0.81      0.69      0.74       124

     accuracy                           0.76       698
    macro avg       0.75      0.58      0.62       698
 weighted avg       0.76      0.76      0.73       698



SVM with Tfidf - linear kernel
               precision    recall  f1-score   support

     business       0.65      0.58      0.61       101
entertainment       0.82      0.94      0.87       400
        other       0.74      0.47      0.57        73
     politics       0.91      0.74      0.82       124

     accuracy                           0.80       698
    macro avg       0.78      0.68      0.72       698
 weighted avg       0.80      0.80      0.80       698



LogReg with Count + spaCy lemmatizer


c:\Users\stolw010\AppData\Local\anaconda3\envs\gesis_iml\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\stolw010\AppData\Local\anaconda3\envs\gesis_iml\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\stolw010\AppData\Local\anaconda3\envs\gesis_iml\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


               precision    recall  f1-score   support

     business       0.60      0.47      0.53       101
entertainment       0.77      0.95      0.85       400
        other       0.81      0.23      0.36        73
     politics       0.84      0.70      0.76       124

     accuracy                           0.76       698
    macro avg       0.75      0.59      0.63       698
 weighted avg       0.76      0.76      0.74       698



SVM with Tfidf - linear kernel + spaCy lemmatizer
               precision    recall  f1-score   support

     business       0.64      0.58      0.61       101
entertainment       0.82      0.94      0.88       400
        other       0.71      0.44      0.54        73
     politics       0.90      0.75      0.82       124

     accuracy                           0.80       698
    macro avg       0.77      0.68      0.71       698
 weighted avg       0.80      0.80      0.79       698





#### Note the new solver ('saga') selected for LogisticRegression had problems converging, which increased the time needed to run it, and results might be suboptimal (you could use the max_iter parameter (default=100) in LogisticRegression(max_iter=500,...) to allow more runs to try to converge)

#### also note its poorer performance with respect to the depricated solver ('liblinear') above

#### The complex spacy lemmatizer also adds substantively to the processing time -> looking at the results, is it worth it?

In [20]:
#remember we didn't save the predictions of the best model, so let's do that now it was cleared after the end of the iteration...
#need to re-calculate the predictions of the best model

#compare the predicted labels of the best model to the true labels
description, vectorizer, classifier =  [('SVM with Tfidf - linear kernel', TfidfVectorizer(min_df=5, max_df=0.5), SVC(kernel='linear', probability=True))][0]

X_tr = vectorizer.fit_transform(X_train)
X_te = vectorizer.transform(X_test)
classifier.fit(X_tr, y_train)
y_pred = classifier.predict(X_te)

#now save this in a pd.DataFrame
import pandas as pd
results_df = pd.DataFrame({'text': X_test, 'true_label': y_test, 'predicted_label': y_pred})
#add a column whether the prediction was correct
results_df['correct'] = results_df['true_label'] == results_df['predicted_label']

#show the first few incorrect predictions per class:
for label in results_df['true_label'].unique():
    incorrect_predictions = results_df[(results_df['true_label'] == label) & (results_df['correct'] == False)]
    print(incorrect_predictions[['text', 'true_label', 'predicted_label']].head())

#get predicted probabilities for the best model
if hasattr(classifier, "predict_proba"):
    y_proba = classifier.predict_proba(X_te)
    #get the max probability for each prediction
    max_proba = y_proba.max(axis=1)
    results_df['max_proba'] = max_proba
    #show the first few incorrect predictions with the highest confidence
    incorrect_predictions = results_df[results_df['correct'] == False]
    high_confidence_incorrect = incorrect_predictions.sort_values(by='max_proba', ascending=False)
    print("High confidence incorrect predictions:\n", high_confidence_incorrect.loc[:, ['text', 'true_label', 'predicted_label', 'max_proba']].head())

                                                 text true_label  \
4   De gemeente Amsterdam ligt onder vuur wegens h...   politics   
31  De provincie, hoogheemraadschap en andere over...   politics   
59  De oorlog is nooit ver weg in Nagorno-Karabach...   politics   
71  Al lang voor hij president was deden er een ho...   politics   
82  Boven: Twee rabbi’s zegenen dinsdag in Berlijn...   politics   

   predicted_label  
4    entertainment  
31   entertainment  
59   entertainment  
71   entertainment  
82   entertainment  
                                                 text true_label  \
3   Gebiedsontwikkelaar Chipshol heeft weer een de...   business   
8   pagina 10 - 11  Amsterdam.  De drie Nederlands...   business   
14  ROERMOND/TILBURG - Rijkswaterstaat is begonnen...   business   
33  De gemeente Almere en Tactus Verslavingszorg g...   business   
38  Dit schrijft De Telegraaf zaterdag op basis va...   business   

   predicted_label  
3    entertainment  
8    entertain

In [21]:
#make a pd.dataframe with the text, true label, predicted label, whether the prediction was correct, and the predicted probability for each label

#first get the predicted probabilities for each label
if hasattr(classifier, "predict_proba"):
    y_proba = classifier.predict_proba(X_te)
    #get the class labels
    class_labels = classifier.classes_
    #create a dataframe with the predicted probabilities for each label
    proba_df = pd.DataFrame(y_proba, columns=[f'proba_{label}' for label in class_labels])
    #concatenate this with the results_df
    results_df = pd.concat([results_df, proba_df], axis=1)  
    print(results_df.head())


                                                text     true_label  \
0  artikelVoor Thierry Baudets partij staat Yerna...       politics   
1  In de Griekse hoofdstad Athene worden heel vee...       business   
2  Vrijdag, 12 januari 2018 om 13:31  Stefan de V...  entertainment   
3  Gebiedsontwikkelaar Chipshol heeft weer een de...       business   
4  De gemeente Amsterdam ligt onder vuur wegens h...       politics   

  predicted_label  correct  max_proba  proba_business  proba_entertainment  \
0        politics     True   0.982807        0.006398             0.008356   
1        business     True   0.593461        0.593461             0.261821   
2   entertainment     True   0.991800        0.003939             0.991800   
3   entertainment    False   0.584540        0.584540             0.374492   
4   entertainment    False   0.893843        0.013390             0.893843   

   proba_other  proba_politics  
0     0.002438        0.982807  
1     0.099852        0.044865  
2    

#### OPTIONAL AND ADVANCED use the explainable AI package Shapley to help understand the model

### WARNING EXTREMELY SLOW EVEN ON A SMALL SAMPLE

In [4]:
%pip install shap

   ---------------------------------------- 0.0/544.3 kB ? eta -:--:--
   ---------------------------------------- 544.3/544.3 kB 6.0 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [slicer]
   ------------- -------------------------- 1/3 [cloudpickle]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -----------

In [ ]:
import shap

#pick the best model:
description, vectorizer, classifier =  [('SVM with Tfidf - linear kernel', TfidfVectorizer(min_df=5, max_df=0.5), SVC(kernel='linear', probability=True))][0]

X_tr = vectorizer.fit_transform(X_train)
X_te = vectorizer.transform(X_test)

# Fit your model as usual
classifier.fit(X_tr, y_train)

# Use the model's predict_proba for SHAP
explainer = shap.KernelExplainer(classifier.predict_proba, X_tr[:10])  # use a sample for background
shap_values = explainer.shap_values(X_te[:10])  # explain a few test samples

# Visualize
shap.summary_plot(shap_values, X_te[:10])